# Week 04 — Model Evaluation
**Course:** Machine Learning & Deep Learning Project  
**Dataset:** MovieLens (Movie Recommendation System - Classification Framing)  

---

### SOP Task List (Verbatim):
> **Week 4: Model Evaluation**
> - [x] Evaluate model over test data.
> - [x] Check model is overfit, underfit or perfect.


## Week Objective
We will evaluate our saved Logistic Regression model on the unseen **Testing dataset (20%)**. We will engineer the same features used during training, calculate classification metrics (Accuracy, Precision, Recall, F1-Score), and finally compare Training vs. Testing performance to definitively diagnose whether the model suffers from **overfitting, underfitting, or is a good fit**.


In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from IPython.display import display

# Robust path resolution
base_parent = ".." if os.path.exists(os.path.join("..", "CSVs")) else "."
def resolve_file(rel_path):
    for candidate in [os.path.join("..", rel_path), rel_path]:
        if os.path.exists(candidate):
            return candidate
    return os.path.join(base_parent, rel_path)

TRAIN_PATH = resolve_file(os.path.join("data", "processed", "train.csv"))
TEST_PATH = resolve_file(os.path.join("data", "processed", "test.csv"))
MODEL_PATH = resolve_file(os.path.join("models", "logistic_regression_lib.joblib"))

# Random Seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment configured successfully.")
print(f"Train Path: {TRAIN_PATH}")
print(f"Test Path:  {TEST_PATH}")
print(f"Model Path: {MODEL_PATH}")



Environment configured successfully.


---
## 1. Test Data Preparation & Feature Engineering
To prevent data leakage, we must engineer the `user_avg_rating` and `movie_avg_rating` features for the Test set using **ONLY the historical averages calculated from the Training set**. 
If a user or movie in the test set is completely new (Cold Start Problem), we will impute their average with the global training average.


In [2]:
# Load datasets
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Define target variable (1 = Liked (>=4.0), 0 = Not Liked)
train_df['is_liked'] = (train_df['rating'] >= 4.0).astype(int)
test_df['is_liked'] = (test_df['rating'] >= 4.0).astype(int)

# Calculate averages from TRAIN ONLY
user_avgs = train_df.groupby('userId')['rating'].mean().to_dict()
movie_avgs = train_df.groupby('movieId')['rating'].mean().to_dict()
global_avg = train_df['rating'].mean()

# Map averages to datasets
# For Training Data
train_df['user_avg_rating'] = train_df['userId'].map(user_avgs)
train_df['movie_avg_rating'] = train_df['movieId'].map(movie_avgs)

# For Testing Data (with fallback to global average for unseen entities)
test_df['user_avg_rating'] = test_df['userId'].map(user_avgs).fillna(global_avg)
test_df['movie_avg_rating'] = test_df['movieId'].map(movie_avgs).fillna(global_avg)

# Prepare X and y matrices
X_train = train_df[['user_avg_rating', 'movie_avg_rating']].values
y_train = train_df['is_liked'].values

X_test = test_df[['user_avg_rating', 'movie_avg_rating']].values
y_test = test_df['is_liked'].values

print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}")


Training Features Shape: (826848, 2)
Testing Features Shape: (206713, 2)


---
## 2. Evaluate Model Over Test Data
We load the `scikit-learn` Logistic Regression model saved in Week 3 and evaluate it using standard classification metrics.


In [3]:
# Load the trained model
model = joblib.load(MODEL_PATH)

# Predict on Test Data
y_pred_test = model.predict(X_test)

# Calculate Metrics
test_acc = accuracy_score(y_test, y_pred_test)
test_prec = precision_score(y_test, y_pred_test)
test_rec = recall_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)

print(f"--- Test Set Performance ---")
print(f"Accuracy:  {test_acc * 100:.2f}%")
print(f"Precision: {test_prec * 100:.2f}%")
print(f"Recall:    {test_rec * 100:.2f}%")
print(f"F1-Score:  {test_f1 * 100:.2f}%")


--- Test Set Performance ---
Accuracy:  70.66%
Precision: 71.26%
Recall:    77.27%
F1-Score:  74.14%


---
## 3. Overfitting, Underfitting, or Perfect Fit Analysis
To diagnose the model's generalization capabilities, we must compare its performance on the Training set against the Testing set.

- **Overfitting:** High Training Accuracy, Low Testing Accuracy (High Variance).
- **Underfitting:** Low Training Accuracy, Low Testing Accuracy (High Bias).
- **Good Fit:** High Training Accuracy, High Testing Accuracy (Small Delta).


In [4]:
# Predict on Training Data
y_pred_train = model.predict(X_train)

# Calculate Training Metrics
train_acc = accuracy_score(y_train, y_pred_train)
train_prec = precision_score(y_train, y_pred_train)
train_rec = recall_score(y_train, y_pred_train)
train_f1 = f1_score(y_train, y_pred_train)

# Create Comparison Table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Training Set (%)': [train_acc*100, train_prec*100, train_rec*100, train_f1*100],
    'Testing Set (%)': [test_acc*100, test_prec*100, test_rec*100, test_f1*100]
})

comparison_df['Delta (Train - Test)'] = comparison_df['Training Set (%)'] - comparison_df['Testing Set (%)']

print("--- Train vs Test Generalization Check ---")
display(comparison_df.round(2))


--- Train vs Test Generalization Check ---


,Metric,Training Set (%),Testing Set (%),Delta (Train - Test)
0,Accuracy,70.86,70.66,0.20
1,Precision,71.48,71.26,0.23
2,Recall,77.28,77.27,0.01
3,F1-Score,74.27,74.14,0.13


### Diagnostic Conclusion
Based on the table above:
1. The **Delta** between Training and Testing metrics is extremely small (less than 1%).
2. This indicates the model is **NOT Overfitting**. It generalizes perfectly to unseen data.
3. The overall accuracy is reasonably high for a basic linear model using only two feature dimensions, meaning it is **NOT Underfitting** either.
4. **Verdict:** The model is a **Good/Perfect Fit** given the constraints of linear classification.


---
## ✅ Week 4 Completion Checklist

- [x] Processed test data ensuring absolutely zero data leakage (used train averages).
- [x] Evaluated Logistic Regression over test data.
- [x] Calculated comprehensive metrics (Accuracy, Precision, Recall, F1).
- [x] Checked model for Over/Underfitting by comparing Train vs. Test metrics.
- [x] Provided diagnostic conclusion.
